In [6]:
import os
import shutil
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_ollama import OllamaLLM
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

# ── Paths ──────────────────────────────────────
docs_path = os.path.expanduser("~/sports-rag-chatbot/documents")
db_path   = os.path.expanduser("~/sports-rag-chatbot/chroma_db")

# ── Clear old DB ───────────────────────────────
if os.path.exists(db_path):
    shutil.rmtree(db_path)
print("✅ DB cleared")

# ── Load documents ─────────────────────────────
loader    = DirectoryLoader(docs_path, glob="**/*.txt", loader_cls=TextLoader)
documents = loader.load()
print(f"✅ Loaded {len(documents)} documents")

# ── Chunk ──────────────────────────────────────
splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
    separators=["\n\n", "\n", ".", " "]
)
chunks = splitter.split_documents(documents)
print(f"✅ Created {len(chunks)} chunks")

# ── Embeddings ─────────────────────────────────
embeddings = HuggingFaceEmbeddings(
    model_name="all-MiniLM-L6-v2",
    model_kwargs={"device": "cpu"}
)

# ── Vector Store ───────────────────────────────
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory=db_path
)
print(f"✅ Vector store created with {vectorstore._collection.count()} chunks!")

# ── Retriever ──────────────────────────────────
retriever = vectorstore.as_retriever(
    search_type="mmr",
    search_kwargs={"k": 6, "fetch_k": 12}
)

# ── LLM ───────────────────────────────────────
llm = OllamaLLM(model="llama3.2", temperature=0.1)

# ── Prompt ────────────────────────────────────
prompt_template = """You are a world-class sports analytics expert with deep knowledge of football, basketball, NFL, cricket, tennis, Formula 1, and all major sports worldwide.

Use the following context to answer the question accurately and enthusiastically.
If the answer is not in the context say "I don't have that information in my knowledge base."
Always cite specific statistics, scores and facts when available.
Keep answers concise but informative.

Context:
{context}

Question: {question}

Answer:"""

prompt = PromptTemplate(
    template=prompt_template,
    input_variables=["context", "question"]
)

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# ── RAG Chain ─────────────────────────────────
qa_chain = (
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough()
    }
    | prompt
    | llm
    | StrOutputParser()
)

print("✅ Full RAG pipeline ready!")

✅ DB cleared
✅ Loaded 10 documents
✅ Created 57 chunks


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

InternalError: Database error: error returned from database: (code: 14) unable to open database file

In [4]:
test_questions = [
    "Who won the 2024 Ballon d'Or and why was it historic?",
    "What happened when Mbappe joined Real Madrid?",
    "Who won the 2024 Super Bowl?",
    "Compare Messi and Ronaldo's Champions League goals",
    "Who is Jannik Sinner and why is he important in tennis?",
    "Tell me about the Boston Celtics 2024 championship",
    "Who won the ICC T20 World Cup 2024?",
    "What happened at the 2024 Paris Olympics basketball final?",
]

for question in test_questions:
    print(f"\n{'='*60}")
    print(f"Q: {question}")
    print(f"{'='*60}")
    answer = qa_chain.invoke(question)
    print(f"A: {answer}")


Q: Who won the 2024 Ballon d'Or and why was it historic?
A: What an exciting year for sports!

Rodri of Manchester City won the 2024 Ballon d'Or award, marking a historic moment where neither Lionel Messi nor Cristiano Ronaldo took home the top prize since 2018. This achievement is significant because Rodri's outstanding performances for both Manchester City and Spain made him stand out from the rest.

Specifically, his key role in Spain's Euro 2024 victory as a midfielder was instrumental in their success. The Spanish team's win marked their first major international title in 12 years, with Rodri being a crucial player in their journey to the championship.

This historic win also highlights the growing talent pool in European football, with players like Rodri and Vinicius Junior (who finished second) showcasing their skills on the global stage.

Q: What happened when Mbappe joined Real Madrid?
A: When Kylian Mbappe joined Real Madrid in summer 2024, he ended years of speculation abou

In [7]:
# Check what documents are in the DB
docs = retriever.invoke("Who won the 2024 Ballon d'Or?")
print(f"Retrieved {len(docs)} chunks:\n")
for i, doc in enumerate(docs):
    print(f"--- Chunk {i+1} ---")
    print(f"Source: {doc.metadata['source'].split('/')[-1]}")
    print(f"Content: {doc.page_content[:300]}")
    print()

InternalError: Error getting collection: Database error: error returned from database: (code: 14) unable to open database file

In [8]:
print("Documents loaded:")
for doc in documents:
    print(f"  - {doc.metadata['source'].split('/')[-1]}")

Documents loaded:
  - cricket.txt
  - nfl_american_football.txt
  - basketball_global.txt
  - tennis.txt
  - premier_league.txt
  - latest_football_2024_2025.txt
  - nba_basketball.txt
  - messi_vs_ronaldo.txt
  - world_cup_history.txt
  - champions_league.txt


In [9]:
import os, shutil
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_chroma import Chroma

db_path = os.path.expanduser("~/sports-rag-chatbot/chroma_db")

# Check if DB exists
print(f"DB exists: {os.path.exists(db_path)}")
print(f"DB files: {os.listdir(db_path) if os.path.exists(db_path) else 'EMPTY'}")

# Reload vectorstore fresh
embeddings = HuggingFaceEmbeddings(
    model_name="all-MiniLM-L6-v2",
    model_kwargs={"device": "cpu"}
)

vectorstore = Chroma(
    persist_directory=db_path,
    embedding_function=embeddings
)

print(f"Chunks in DB: {vectorstore._collection.count()}")

# Test retrieval
retriever = vectorstore.as_retriever(search_kwargs={"k": 6})
docs = retriever.invoke("Who won the 2024 Ballon d'Or?")

print(f"\nRetrieved {len(docs)} chunks:")
for i, doc in enumerate(docs):
    print(f"\n--- Chunk {i+1} ---")
    print(f"Source: {doc.metadata['source'].split('/')[-1]}")
    print(f"Content: {doc.page_content[:200]}")

DB exists: False
DB files: EMPTY


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

InternalError: Database error: error returned from database: (code: 14) unable to open database file